# Shared-layer SNF standard, beta protocol, lattice 2^4

Notebook di test per confrontare tre protocolli standard in beta, senza difetto e senza `DefectCouplingLayer`:

- protocollo puramente stocastico;
- SNF residual con un solo `ResidualCouplingLayer` riusato a ogni update deterministico;
- SNF smearing con un solo `CouplingLayer + HyperSmearing` riusato a ogni update deterministico.

I valori `beta_prior`, `beta_target` e `n_layers` sono definiti nella cella di configurazione.

In [51]:
import sys
from pathlib import Path

import torch

root = Path.cwd()
if (root / 'src').exists():
    project_root = root
elif (root.parent / 'src').exists():
    project_root = root.parent
else:
    raise RuntimeError('Non trovo la cartella src')

sys.path.insert(0, str(project_root / 'src'))

import neoqcd.sun_utils as sun
from neoqcd.flow import Flow, FlowPars
from neoqcd.mcmc import NEMCMC_update
from neoqcd.smearing import CouplingLayer, ResidualCouplingLayer
from neoqcd.theory import PriorSUN, Wilson_action
from neoqcd.utils import create_mask

torch.manual_seed(1234)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = torch.float64

D = 4
T = 4
L = 4
N = 3
batch_size = 16

beta_prior = 1.0
beta_target = 1.5
n_layers = 4

therm_steps = 100
mcmc_steps = 1
train_epochs = 10
eval_batches = 16

lr = 2e-3
grad_clip = 10.0

print(f'device={device}, lattice={T}x{L}x{L}x{L}, batch={batch_size}')


device=cpu, lattice=4x4x4x4, batch=16


In [52]:
mask = create_mask(D, T, L).to(device)
protocol = torch.linspace(beta_prior, beta_target, n_layers, dtype=dtype).tolist()


def make_flow_pars(
    nf_layer_type,
    rho_shape_type=0,
    use_hyper_smearing=False,
    hyper_smearing_mode='shared',
    hyper_scale_by_delta=False,
):
    return FlowPars(
        D=D,
        T=T,
        L=L,
        N=N,
        mask=mask,
        protocol=protocol,
        batch_size=batch_size,
        device=device,
        rho_shape_type=rho_shape_type,
        set_rho=0,
        fit_train=0,
        orsteps=0,
        updates_per_layer=1,
        smearing_steps_per_layer=1,
        nf_layer_type=nf_layer_type,
        use_hyper_smearing=use_hyper_smearing,
        hyper_smearing_mode=hyper_smearing_mode,
        hyper_time_embedding_dim=8,
        hyper_hidden_dim=16,
        hyper_depth=2,
        hyper_activation='silu',
        hyper_rho_init=1e-3,
        hyper_normalize_by_nstep=False,
        hyper_rho_eps=0.0,
        hyper_scale_by_delta=hyper_scale_by_delta,
        hyper_rho_max=0.0,
        residual_include_imag=True,
        residual_quadratic=True,
        residual_coeff_init=1e-3,
        residual_coeff_max=0.0,
    )


residual_flow_pars = make_flow_pars(nf_layer_type='residual')
smearing_flow_pars = make_flow_pars(
    nf_layer_type='smearing',
    rho_shape_type=1,
    use_hyper_smearing=True,
    hyper_smearing_mode='per_link',
    hyper_scale_by_delta=True,
)

prior = PriorSUN(
    flow_pars=residual_flow_pars,
    beta=beta_prior,
    therm_steps=therm_steps,
    mcmc_steps=mcmc_steps,
    defect_mask=None,
    defect_par=1.0,
)
target_action = Wilson_action(residual_flow_pars, beta_target, defect_par=1.0)

with torch.no_grad():
    cfgs, s0 = prior()

print('prior action mean:', float(s0.mean().item()))


prior action mean: 1441.6225180612346


In [53]:
class SharedSNF(torch.nn.Module):
    def __init__(self, flow_pars, layer_type, beta=1.0, defect_mask=None):
        super().__init__()
        self.flow_pars = flow_pars
        self.layer_type = str(layer_type)
        self.defect_mask = defect_mask
        rho0 = torch.zeros(flow_pars.rho_shape, dtype=torch.float64, device=flow_pars.device)

        if self.layer_type == 'residual':
            self.nf_layer = ResidualCouplingLayer(flow_pars, rho0, time=1.0)
        elif self.layer_type == 'smearing':
            self.nf_layer = CouplingLayer(flow_pars, rho0, time=1.0)
        else:
            raise ValueError(f'unknown layer_type={layer_type!r}')

        self.stochastic_layers = torch.nn.ModuleList(
            [NEMCMC_update(flow_pars, beta_step) for beta_step in flow_pars.protocol]
        )

    def _beta_delta(self, step):
        nstep = int(self.flow_pars.nstep)
        idx = int(max(0, min(step, nstep - 1)))
        beta_step = float(self.flow_pars.protocol[idx])
        if nstep == 1:
            delta_step = 0.0
        elif idx < nstep - 1:
            delta_step = float(self.flow_pars.protocol[idx + 1] - self.flow_pars.protocol[idx])
        else:
            delta_step = float(self.flow_pars.protocol[idx] - self.flow_pars.protocol[idx - 1])
        return beta_step, delta_step

    def _run_nf_block(self, x, step):
        beta_step, delta_step = self._beta_delta(step)
        return self.nf_layer.forward(
            x,
            self.flow_pars,
            rho_layer=None,
            beta=beta_step,
            delta_beta=delta_step,
        )

    def _run_stochastic_block(self, x, step):
        return self.stochastic_layers[int(step)].forward(x, self.flow_pars, rho_root=None)

    def forward_up_to_block(self, x, tb):
        work_dtype = x.real.dtype if torch.is_complex(x) else x.dtype
        Q = torch.zeros(x.shape[0], dtype=work_dtype, device=x.device)
        logJ = torch.zeros(x.shape[0], dtype=work_dtype, device=x.device)
        for step in range(int(tb)):
            x, dQ, dlogJ = self._run_nf_block(x, step)
            Q = Q + dQ
            logJ = logJ + dlogJ
            x, dQ, dlogJ = self._run_stochastic_block(x, step)
            Q = Q + dQ
            logJ = logJ + dlogJ
        return x, Q, logJ

    def up_to_block_nograd(self, x, s0, tb):
        with torch.no_grad():
            x, Q, logJ = self.forward_up_to_block(x, tb)
        x, dQ, dlogJ = self._run_nf_block(x, tb)
        Q = Q + dQ
        logJ = logJ + dlogJ
        st = self.stochastic_layers[int(tb)].action(x, self.defect_mask)
        w = st - s0 - Q - 2.0 * logJ
        return x, w, st, Q, 2.0 * logJ

    def forward(self, x, s0, int_meas=1):
        batch_size = int(x.shape[0])
        work_dtype = x.real.dtype if torch.is_complex(x) else x.dtype
        Q = torch.zeros(batch_size, dtype=work_dtype, device=x.device)
        logJ = torch.zeros(batch_size, dtype=work_dtype, device=x.device)
        int_obs = torch.zeros((batch_size, int_meas, 5), dtype=work_dtype, device=x.device)
        nstep_per_meas = max(1, int(self.flow_pars.nstep) // int_meas)
        m = 0

        for step in range(int(self.flow_pars.nstep)):
            x, dQ, dlogJ = self._run_nf_block(x, step)
            Q = Q + dQ
            logJ = logJ + dlogJ
            x, dQ, dlogJ = self._run_stochastic_block(x, step)
            Q = Q + dQ
            logJ = logJ + dlogJ

            if (step + 1) % nstep_per_meas == 0 and m < int_meas:
                st_m = self.stochastic_layers[step].action(x, self.defect_mask)
                int_obs[:, m, 0] = st_m - s0 - Q - 2.0 * logJ
                int_obs[:, m, 1] = Q
                int_obs[:, m, 2] = 2.0 * logJ
                int_obs[:, m, 3] = st_m
                int_obs[:, m, 4] = float(self.flow_pars.protocol[step])
                m += 1

        st = self.stochastic_layers[-1].action(x, self.defect_mask)
        w = st - s0 - Q - 2.0 * logJ
        return x, w, st, Q, 2.0 * logJ, int_obs

    def compute_metrics(self, w):
        wd = w.detach()
        wdm = wd.mean()
        expwm = torch.mean(torch.exp(-(wd - wdm)))
        delta_f = -torch.log(expwm) + wdm
        ess = (expwm ** 2) / torch.mean(torch.exp(-2.0 * (wd - wdm)))
        return delta_f, ess


class PureStochasticProtocol(torch.nn.Module):
    def __init__(self, flow_pars, beta=1.0, defect_mask=None):
        super().__init__()
        self.flow_pars = flow_pars
        self.flow = Flow(
            flow_pars,
            flow_type='nemcmc',
            protocol_type='beta',
            beta=beta,
            defect_mask=defect_mask,
        )

    def forward(self, x, s0, int_meas=1):
        return self.flow(x, s0, self.flow_pars, int_meas=int_meas)

    def compute_metrics(self, w):
        return self.flow.compute_metrics(w)


In [54]:
def max_su3_error(U):
    eye = torch.eye(N, dtype=U.dtype, device=U.device)
    unit_err = (sun.SUN_mul(sun.SUN_dagger(U), U) - eye).abs().max()
    det_err = (sun.SUN_determinant(U) - 1.0).abs().max()
    return max(float(unit_err.item()), float(det_err.item()))


def work_summary(model, work, target_actions, su3_errors):
    work_all = work if torch.is_tensor(work) else torch.cat(work).to(device=device, dtype=dtype)
    target_all = target_actions if torch.is_tensor(target_actions) else torch.cat(target_actions).to(device=device, dtype=dtype)
    delta_f, ess = model.compute_metrics(work_all)
    work_mean = work_all.mean()
    dkl = work_mean - delta_f
    return {
        'n_samples': int(work_all.numel()),
        'work_mean': float(work_mean.item()),
        'work_std': float(work_all.std(unbiased=False).item()),
        'delta_f': float(delta_f.item()),
        'dkl': float(dkl.item()),
        'ess': float(ess.item()),
        'target_action_mean': float(target_all.mean().item()),
        'su3_error_max': float(max(su3_errors)),
    }


def evaluate_model_once(model, x, s):
    model.eval()
    with torch.no_grad():
        x_out, work, st, heat, logj, int_obs = model(x.detach(), s.detach(), int_meas=4)
        return work_summary(model, work, target_action(x_out, None), [max_su3_error(x_out)])


def train_local_layers_for(model, optimizer, x, s):
    model.train()
    rows = []
    for tb in range(n_layers):
        optimizer.zero_grad(set_to_none=True)
        _, work_layer, st_layer, _, logj_layer = model.up_to_block_nograd(
            x.detach(),
            s.detach(),
            tb,
        )
        loss = work_layer.mean()
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        rows.append(
            {
                'layer': tb,
                'loss': float(loss.detach().item()),
                'work_mean': float(work_layer.detach().mean().item()),
                'action_mean': float(st_layer.detach().mean().item()),
                'logj_mean': float(logj_layer.detach().mean().item()),
                'grad_norm': float(torch.as_tensor(grad_norm).detach().cpu().item()),
            }
        )
    return rows


def train_model(model, optimizer, label):
    history = []
    x_train = cfgs.detach().clone()
    s_train = s0.detach().clone()
    for epoch in range(train_epochs):
        with torch.no_grad():
            x_train, s_train = prior(x_train.detach())

        local_rows = train_local_layers_for(model, optimizer, x_train, s_train)
        metrics = evaluate_model_once(model, x_train, s_train)
        history.append({'epoch': epoch, 'local_rows': local_rows, 'metrics': metrics})

        print(
            f"{label} epoch={epoch:02d} "
            f"local_loss_mean={sum(r['loss'] for r in local_rows) / len(local_rows):.6g} "
            f"work={metrics['work_mean']:.6g}+/-{metrics['work_std']:.6g} "
            f"DeltaF={metrics['delta_f']:.6g} "
            f"DKL={metrics['dkl']:.6g} "
            f"ess={metrics['ess']:.6g} "
            f"su3_err={metrics['su3_error_max']:.3e}"
        )
    return history


## Build shared SNF models

Ogni modello SNF contiene un solo layer deterministico, riusato per tutti gli update deterministici del protocollo.

In [55]:
residual_flow = SharedSNF(residual_flow_pars, layer_type='residual', beta=beta_prior).to(device)
smearing_flow = SharedSNF(smearing_flow_pars, layer_type='smearing', beta=beta_prior).to(device)
pure_stochastic_flow = PureStochasticProtocol(residual_flow_pars, beta=beta_prior).to(device)

residual_optimizer = torch.optim.Adam(residual_flow.parameters(), lr=lr)
smearing_optimizer = torch.optim.Adam(smearing_flow.parameters(), lr=lr)

assert isinstance(residual_flow.nf_layer, ResidualCouplingLayer)
assert isinstance(smearing_flow.nf_layer, CouplingLayer)
assert getattr(smearing_flow.nf_layer.smearing, 'use_hyper_smearing', False)
assert smearing_flow.nf_layer.smearing.hyper_smearing.mode == 'per_link'
assert smearing_flow.nf_layer.smearing.hyper_smearing.scale_by_delta

print('residual shared layer:', type(residual_flow.nf_layer).__name__)
print('smearing shared layer:', type(smearing_flow.nf_layer).__name__, '+ HyperSmearing')


residual shared layer: ResidualCouplingLayer
smearing shared layer: CouplingLayer + HyperSmearing


## Training: residual shared SNF

In [56]:
residual_history = train_model(residual_flow, residual_optimizer, label='residual')
residual_history[-1]


residual epoch=00 local_loss_mean=357.945 work=713.476+/-2.94079 DeltaF=710.923 DKL=2.55349 ess=0.156799 su3_err=1.313e-14
residual epoch=01 local_loss_mean=357.318 work=711.772+/-1.94057 DeltaF=710.273 DKL=1.49915 ess=0.271984 su3_err=1.355e-14
residual epoch=02 local_loss_mean=356.98 work=710.716+/-1.90266 DeltaF=708.68 DKL=2.03596 ess=0.121949 su3_err=1.333e-14
residual epoch=03 local_loss_mean=356.729 work=710.213+/-1.50778 DeltaF=709.077 DKL=1.13685 ess=0.239292 su3_err=1.221e-14
residual epoch=04 local_loss_mean=356.759 work=710.566+/-1.60326 DeltaF=709.56 DKL=1.00568 ess=0.392291 su3_err=1.221e-14
residual epoch=05 local_loss_mean=357.045 work=710.477+/-1.28115 DeltaF=709.912 DKL=0.565403 ess=0.524323 su3_err=1.199e-14
residual epoch=06 local_loss_mean=356.456 work=709.702+/-1.14305 DeltaF=709.151 DKL=0.550585 ess=0.499522 su3_err=1.355e-14
residual epoch=07 local_loss_mean=356.805 work=709.994+/-1.10794 DeltaF=709.176 DKL=0.817529 ess=0.216317 su3_err=1.310e-14
residual epoch=0

{'epoch': 9,
 'local_rows': [{'layer': 0,
   'loss': 0.4947761988860207,
   'work_mean': 0.4947761988860207,
   'action_mean': 1435.2278728951865,
   'logj_mean': -6.9152066651219055,
   'grad_norm': 7.977825106855293},
  {'layer': 1,
   'loss': 239.89247094334925,
   'work_mean': 239.89247094334925,
   'action_mean': 1672.7494101765965,
   'logj_mean': -18.435215715597128,
   'grad_norm': 5.609051607175312},
  {'layer': 2,
   'loss': 476.3636444314027,
   'work_mean': 476.3636444314027,
   'action_mean': 1885.8583409771118,
   'logj_mean': -37.77314158665403,
   'grad_norm': 4.803012213201388},
  {'layer': 3,
   'loss': 710.0224481723637,
   'work_mean': 710.0224481723637,
   'action_mean': 2097.246615453858,
   'logj_mean': -55.285185774788424,
   'grad_norm': 6.402849552791139}],
 'metrics': {'n_samples': 16,
  'work_mean': 709.6666486740419,
  'work_std': 0.995096089565902,
  'delta_f': 709.123007943191,
  'dkl': 0.5436407308508251,
  'ess': 0.3865292991210071,
  'target_action_mea

## Training: hyper-smearing shared SNF

In [57]:
smearing_history = train_model(smearing_flow, smearing_optimizer, label='smearing')
smearing_history[-1]


smearing epoch=00 local_loss_mean=357.662 work=712.12+/-3.2127 DeltaF=708.527 DKL=3.59283 ess=0.145687 su3_err=1.244e-14
smearing epoch=01 local_loss_mean=357.465 work=710.979+/-1.10977 DeltaF=710.418 DKL=0.561142 ess=0.391636 su3_err=1.200e-14
smearing epoch=02 local_loss_mean=356.819 work=709.94+/-1.14111 DeltaF=709.106 DKL=0.834859 ess=0.22398 su3_err=1.188e-14
smearing epoch=03 local_loss_mean=356.776 work=710.135+/-1.32358 DeltaF=709.249 DKL=0.88582 ess=0.31762 su3_err=1.377e-14
smearing epoch=04 local_loss_mean=356.982 work=710.304+/-1.46957 DeltaF=709.099 DKL=1.20573 ess=0.163702 su3_err=1.466e-14
smearing epoch=05 local_loss_mean=357.262 work=710.808+/-1.43575 DeltaF=709.559 DKL=1.24876 ess=0.163893 su3_err=1.222e-14
smearing epoch=06 local_loss_mean=357.288 work=710.316+/-0.742385 DeltaF=710.111 DKL=0.205601 ess=0.786125 su3_err=1.466e-14
smearing epoch=07 local_loss_mean=356.752 work=710.129+/-1.2073 DeltaF=709.477 DKL=0.651743 ess=0.44063 su3_err=1.422e-14
smearing epoch=08 

{'epoch': 9,
 'local_rows': [{'layer': 0,
   'loss': 0.4234757010188189,
   'work_mean': 0.4234757010188189,
   'action_mean': 1434.6885132769914,
   'logj_mean': -8.623231355116591,
   'grad_norm': 9.474327702673799},
  {'layer': 1,
   'loss': 239.7962403567259,
   'work_mean': 239.7962403567259,
   'action_mean': 1673.306364224531,
   'logj_mean': -19.13866789735311,
   'grad_norm': 10.355886526501852},
  {'layer': 2,
   'loss': 476.2556515223078,
   'work_mean': 476.2556515223078,
   'action_mean': 1882.3704684732375,
   'logj_mean': -37.932765652779686,
   'grad_norm': 8.977497171895312},
  {'layer': 3,
   'loss': 710.0440309746341,
   'work_mean': 710.0440309746341,
   'action_mean': 2092.9644654983194,
   'logj_mean': -59.89103079827463,
   'grad_norm': 8.497880960093704}],
 'metrics': {'n_samples': 16,
  'work_mean': 710.2890956110459,
  'work_std': 0.736643848789611,
  'delta_f': 710.0494512833958,
  'dkl': 0.2396443276501259,
  'ess': 0.7154676220807304,
  'target_action_mean'

## Evaluation: stochastic pure vs trained shared SNF models

La valutazione usa gli stessi batch di prior per i tre modelli, accumula tutti i work e poi stima `<W>`, `DeltaF`, `DKL = <W> - DeltaF`, `ESS`, azione target media e massimo errore SU(3).

In [58]:
def collect_prior_batches(n_batches=eval_batches):
    batches = []
    x_eval = cfgs.detach().clone()
    with torch.no_grad():
        for _ in range(n_batches):
            x_eval, s_eval = prior(x_eval.detach())
            batches.append((x_eval.detach().clone(), s_eval.detach().clone()))
    return batches


def evaluate_in_batches(model, label, prior_batches):
    model.eval()
    work_batches = []
    target_action_batches = []
    su3_errors = []

    with torch.no_grad():
        for x_eval, s_eval in prior_batches:
            x_out, work, st, heat, logj, int_obs = model(x_eval.detach(), s_eval.detach(), int_meas=4)
            work_batches.append(work.detach().cpu())
            target_action_batches.append(target_action(x_out, None).detach().cpu())
            su3_errors.append(max_su3_error(x_out))

    metrics = work_summary(model, work_batches, target_action_batches, su3_errors)
    metrics['label'] = label
    return metrics


eval_prior_batches = collect_prior_batches(eval_batches)

eval_results = {
    'stochastic': evaluate_in_batches(pure_stochastic_flow, 'stochastic', eval_prior_batches),
    'residual_snf': evaluate_in_batches(residual_flow, 'residual_snf', eval_prior_batches),
    'hyper_smearing_snf': evaluate_in_batches(smearing_flow, 'hyper_smearing_snf', eval_prior_batches),
}

for name, metrics in eval_results.items():
    print(
        f"{name:>18s} "
        f"n={metrics['n_samples']} "
        f"work={metrics['work_mean']:.6g}+/-{metrics['work_std']:.6g} "
        f"DeltaF={metrics['delta_f']:.6g} "
        f"DKL={metrics['dkl']:.6g} "
        f"ess={metrics['ess']:.6g} "
        f"target_action={metrics['target_action_mean']:.6g} "
        f"su3_err_max={metrics['su3_error_max']:.3e}"
    )

eval_results


        stochastic n=256 work=713.613+/-2.83877 DeltaF=710.032 DKL=3.58085 ess=0.0433134 target_action=2087.8 su3_err_max=1.310e-14
      residual_snf n=256 work=709.607+/-1.04259 DeltaF=709.108 DKL=0.498528 ess=0.456692 target_action=2088.26 su3_err_max=1.488e-14
hyper_smearing_snf n=256 work=710.002+/-1.21254 DeltaF=709.315 DKL=0.686864 ess=0.278545 target_action=2088.52 su3_err_max=1.488e-14


{'stochastic': {'n_samples': 256,
  'work_mean': 713.6128242614067,
  'work_std': 2.8387666596197367,
  'delta_f': 710.0319782384018,
  'dkl': 3.580846023004824,
  'ess': 0.04331341826088288,
  'target_action_mean': 2087.8016608189396,
  'su3_error_max': 1.310185501178063e-14,
  'label': 'stochastic'},
 'residual_snf': {'n_samples': 256,
  'work_mean': 709.6068219009965,
  'work_std': 1.0425917357507406,
  'delta_f': 709.1082942871961,
  'dkl': 0.49852761380032007,
  'ess': 0.4566915036799583,
  'target_action_mean': 2088.2634199112526,
  'su3_error_max': 1.4883989397448344e-14,
  'label': 'residual_snf'},
 'hyper_smearing_snf': {'n_samples': 256,
  'work_mean': 710.0016121295873,
  'work_std': 1.2125364091348811,
  'delta_f': 709.3147485509719,
  'dkl': 0.68686357861543,
  'ess': 0.2785445748089994,
  'target_action_mean': 2088.5202817552035,
  'su3_error_max': 1.4877487154271324e-14,
  'label': 'hyper_smearing_snf'}}